
# Hansard RAG — indexing

Embeds the chunks from `ingest.ipynb` and indexes them into Elasticsearch, which gives us keyword, vector, and hybrid search from one store.

**Before running:** start Elasticsearch in Docker



In [ ]:
import json
from pathlib import Path
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from shared_funcs.search import keyword_search, vector_search, hybrid_search

# CHUNKS_PATH = Path("data/processed/chunks.jsonl")
# INDEX_NAME = "hansard-chunks"

CHUNKS_PATH = Path("data/processed/chunks_contribution.jsonl")
INDEX_NAME = "hansard-chunks-contribution"

EMBED_MODEL = "multi-qa-MiniLM-L6-cos-v1"
EMBED_DIMS = 384

## Load chunks

In [ ]:
chunks = [json.loads(line) for line in CHUNKS_PATH.open()]
print(f"{len(chunks)} chunks")
chunks[0]["debate_title"], chunks[0]["speaker"], chunks[0]["text"][:12]

In [ ]:
chunks[0]["debate_title"], chunks[0]["speaker"], chunks[0]["text"]

## Embedding model

First run downloads the model . Embedding text = chunk text prefixed with the debate title, for context.

In [ ]:
model = SentenceTransformer(EMBED_MODEL)

# builds the string to embed, context prefix so vector lands appropriately
def embedding_text(chunk):
    return f"{chunk['debate_title']}\n{chunk['text']}"


# list of strings to embed, model will loop
texts = [embedding_text(c) for c in chunks]
# batch size for our m2 chip, normalize embeddings keep sthe lenght at 1 cosine similairty then is a dot product 
embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(embeddings.shape)

## Create the index

- `text`, `debate_title`, `speaker` — full-text searchable (keyword search side)
- `party`, `house`, `sitting_date`, `debate_ext_id`, `member_id` — exact-match filter fields
- `text_vector` — dense vector for kNN (cosine; embeddings are pre-normalised)

In [ ]:
es = Elasticsearch("http://localhost:9200")
print(es.info()["version"]["number"])

# schema defintion 
index_settings = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "debate_ext_id": {"type": "keyword"},
            "debate_title": {"type": "text"},
            "sitting_date": {"type": "date"},
            "house": {"type": "keyword"},
            "location": {"type": "keyword"},
            "member_id": {"type": "keyword"},
            "speaker": {"type": "text", "fields": {"raw": {"type": "keyword"}}},
            "constituency": {"type": "keyword"},
            "party": {"type": "keyword"},
            "order_in_section": {"type": "integer"},
            "chunk_index": {"type": "integer"},
            "n_chunks": {"type": "integer"},
            "text": {"type": "text"},
            "hansard_url": {"type": "keyword", "index": False},
            "text_vector": {
                "type": "dense_vector",
                "dims": EMBED_DIMS, # 384
                "index": True,
                "similarity": "cosine",
            },
        }
    },
}

es.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
es.indices.create(index=INDEX_NAME, body=index_settings)
print(f"created index {INDEX_NAME}")

## Index the documents

In [ ]:
from elasticsearch.helpers import bulk

actions = (
    {
        "_index": INDEX_NAME,
        "_id": chunk["chunk_id"],
        **chunk,
        "text_vector": embedding.tolist(),
    }
    for chunk, embedding in zip(chunks, embeddings)
)

# using elastic bulk api instead of one at a time (one hhtp request) 500 per default
n_ok, errors = bulk(es, tqdm(actions, total=len(chunks)))
print(f"indexed {n_ok} docs, {len(errors) if isinstance(errors, list) else errors} errors")
es.indices.refresh(index=INDEX_NAME)
es.count(index=INDEX_NAME)["count"]

## Search functions — the three approaches we'll evaluate

imported from shared_funcs/search.py

1. **Keyword** — BM25 over text + debate title
2. **Vector** — kNN over embeddings
3. **Hybrid** — both, combined with Reciprocal Rank Fusion (RRF)

Each method takes filters which is another evaluation rubric

## Smoke tests

Not the formal evaluation — just eyeballing that each approach returns something sensible and visibly different.

In [ ]:
def show(results):
    for r in results:
        print(f"[{r['sitting_date']}] {r['debate_title']}  —  {r['speaker']}"
              + (f" ({r['party']})" if r.get("party") else ""))
        print(f"   {r['text'][:150]}...")
    print()


query = "NHS hospital repairs and refurbishment"

print("=== keyword ===");  show(keyword_search(query))
print("=== vector ===");   show(vector_search(query))
print("=== hybrid ===");   show(hybrid_search(query))

In [ ]:
# Filtered search: what have Labour speakers said about housing?
show(hybrid_search("housing and homelessness", k=5, filters={"party": "Lab"}))